In [2]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
from scipy import stats
from skimage.io import imread
from skimage.transform import resize
from skimage.color import rgb2gray, rgb2hsv
from skimage.feature import local_binary_pattern, graycomatrix, graycoprops
from skimage.filters import threshold_otsu
from skimage.measure import label, regionprops, shannon_entropy
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from skimage.feature import blob_log
from skimage.filters import gaussian
from sklearn.ensemble import HistGradientBoostingClassifier # The "95% Secret Weapon"
import numpy as np
from scipy import stats
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.svm import SVC
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
# --- 1. FEATURE EXTRACTION ENGINE ---

import numpy as np
import cv2
import os

def extract_features(img_path, size=(128, 128)):
    try:
        # Load image with OpenCV (loads as BGR)
        img_bgr = cv2.imread(img_path)
        if img_bgr is None: return None
        img_bgr = cv2.resize(img_bgr, size)
        
        # ==========================================
        # 1. COLOR NORMALIZATION (Gray World)
        # Fixes the "purple vs red vs whitish" issue
        # ==========================================
        b, g, r = cv2.split(img_bgr)
        m_b, m_g, m_r = np.mean(b), np.mean(g), np.mean(r)
        
        # Find the average gray value of the whole image
        m = (m_b + m_g + m_r) / 3.0
        
        # Prevent division by zero
        if m_b == 0 or m_g == 0 or m_r == 0: return None

        # Scale channels so the average color becomes neutral gray/pink
        b_norm = np.clip(b * (m / m_b), 0, 255).astype(np.uint8)
        g_norm = np.clip(g * (m / m_g), 0, 255).astype(np.uint8)
        r_norm = np.clip(r * (m / m_r), 0, 255).astype(np.uint8)
        img_norm = cv2.merge((b_norm, g_norm, r_norm))
        
        # ==========================================
        # 2. CREATE A CELL MASK
        # ==========================================
        gray_norm = cv2.cvtColor(img_norm, cv2.COLOR_BGR2GRAY)
        # Background is black, so we threshold to find the cell
        _, cell_mask = cv2.threshold(gray_norm, 10, 255, cv2.THRESH_BINARY)
        
        # ==========================================
        # 3. ADAPTIVE SPOT DETECTION
        # ==========================================
        # Parasites show up best as DARK spots in the GREEN channel.
        # adaptiveThreshold looks at a 21x21 pixel neighborhood.
        # It flags a pixel ONLY if it is noticeably darker than its immediate neighbors.
        local_thresh = cv2.adaptiveThreshold(
            g_norm, 
            255, 
            cv2.ADAPTIVE_THRESH_GAUSSIAN_C, 
            cv2.THRESH_BINARY_INV, 
            21,  # Neighborhood size
            5    # Constant subtracted from the mean (controls sensitivity)
        )
        
        # Only keep spots that are INSIDE the cell mask
        actual_spots = cv2.bitwise_and(local_thresh, cell_mask)
        
        # Morphological Opening: Erase tiny single-pixel "dust" artifacts
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
        actual_spots = cv2.morphologyEx(actual_spots, cv2.MORPH_OPEN, kernel)
        
        # ==========================================
        # 4. MEASURE THE SPOTS
        # ==========================================
        # Find continuous blobs of spots
        contours, _ = cv2.findContours(actual_spots, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        num_spots = len(contours)
        
        if num_spots > 0:
            areas = [cv2.contourArea(c) for c in contours]
            max_spot_area = max(areas)
            total_spot_area = sum(areas)
            
            # Find how "Saturated" the spots are (parasites are highly colored)
            img_hsv = cv2.cvtColor(img_norm, cv2.COLOR_BGR2HSV)
            saturation_channel = img_hsv[:, :, 1]
            
            # Mask just the biggest spot to check its color intensity
            c_max = contours[np.argmax(areas)]
            max_spot_mask = np.zeros_like(gray_norm)
            cv2.drawContours(max_spot_mask, [c_max], -1, 255, -1)
            
            spot_saturation = np.mean(saturation_channel[max_spot_mask == 255])
        else:
            max_spot_area = 0
            total_spot_area = 0
            spot_saturation = 0

        # Return our highly robust features
        return np.array([
            num_spots,           # Are there any sharp, localized dots?
            max_spot_area,       # How big is the main dot?
            total_spot_area,     # Are there multiple tiny dots (like image 178)?
            spot_saturation,     # Is the dot a dense color (parasite) or pale (shadow)?
            np.std(gray_norm[cell_mask == 255]) # Overall texture of the cell
        ])

    except Exception as e:
        # print(f"Failed on {img_path}: {e}")
        return None

# --- 2. DATA LOADING ---

def load_data(root_dir):
    X, y = [], []
    class_names = sorted(os.listdir(root_dir))
    
    print(f"Extracting features from {root_dir}...")
    for label_idx, cls_name in enumerate(class_names):
        cls_path = os.path.join(root_dir, cls_name)
        if not os.path.isdir(cls_path): continue
        
        files = [f for f in os.listdir(cls_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        for fname in tqdm(files, desc=f"Class: {cls_name}"):
            f_vector = extract_features(os.path.join(cls_path, fname))
            if f_vector is not None:
                X.append(f_vector)
                y.append(label_idx)
                
    return np.array(X), np.array(y)

    

In [ ]:
import numpy as np
from scipy import stats
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.svm import SVC
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier

# Import Bayesian Optimization
from skopt import BayesSearchCV
from skopt.space import Real, Integer, Categorical

def run_evaluation(root_path):
    # 1. Load and Extract
    X, y = load_data(root_path)
    
    # 2. Define Models and their Bayesian Search Spaces
    # Note: We use 'classifier__' prefix because the model is inside a Pipeline
    models_and_spaces = {
        "SVM": (
            SVC(),
            {
                'classifier__C': Real(1e-1, 1e+2, prior='log-uniform'),
                'classifier__gamma': Categorical(['scale', 'auto']),
                'classifier__kernel': Categorical(['rbf', 'linear'])
            }
        ),
        "HistGradientBoosting": (
            HistGradientBoostingClassifier(),
            {
                'classifier__learning_rate': Real(0.01, 0.2, prior='log-uniform'),
                'classifier__max_depth': Integer(3, 15),
                'classifier__max_iter': Integer(50, 400)
            }
        ),
        "Random Forest": (
            RandomForestClassifier(),
            {
                'classifier__n_estimators': Integer(50, 300),
                'classifier__max_depth': Integer(3, 20),
                'classifier__min_samples_split': Integer(2, 10)
            }
        )
    }
        
    # 3. Setup Cross-Validation
    inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42) # For BayesSearch
    outer_cv = StratifiedKFold(n_splits=20, shuffle=True, random_state=42) # For Stability testing
    
    results = {}


    print("\nStarting Bayesian Optimization & 20-Fold Evaluation...")
    for name, (model, search_space) in models_and_spaces.items():
        print(f"\n[{name}] Optimizing parameters...")
        
        pipeline = Pipeline([
            ('scaler', StandardScaler()),
            ('classifier', model)
        ])
        
        # --- BAYESIAN OPTIMIZATION ---
        # n_iter=15 means it will try 15 highly-educated guesses to find the best params
        bayes_search = BayesSearchCV(
            estimator=pipeline,
            search_spaces=search_space,
            n_iter=15, 
            cv=inner_cv,
            scoring='accuracy',
            random_state=42,
            n_jobs=-1
        )
        
        # Fit it to find the absolute best parameters
        bayes_search.fit(X, y)
        best_pipeline = bayes_search.best_estimator_
        
        print(f"[{name}] Best Params Found: {bayes_search.best_params_}")
        print(f"[{name}] Running 20-Fold CV on optimized model...")
        
        # --- STABILITY EVALUATION ---
        # Run the 20-fold CV using the winning pipeline
        cv_results = cross_validate(best_pipeline, X, y, cv=outer_cv, scoring='accuracy', n_jobs=-1)
        results[name] = cv_results['test_score']

    # 4. STATISTICAL SIGNIFICANCE & STABILITY (95% CI & One-Sample T-Test)
    print("\n=======================================================")
    print("   FINAL OPTIMIZED MODEL STABILITY & 95% CI RESULTS   ")
    print("=======================================================")
    
    baseline_chance = 0.50 
    
    for name, scores in results.items():
        n_folds = len(scores)
        mean_acc = np.mean(scores)
        std_acc = np.std(scores, ddof=1) 
        
        ci_lower, ci_upper = stats.t.interval(
            confidence=0.95, df=n_folds - 1, loc=mean_acc, scale=stats.sem(scores)
        )
        
        t_stat, p_val = stats.ttest_1samp(scores, popmean=baseline_chance, alternative='greater')
        sig = "Stable & Significant" if p_val < 0.05 else "Unstable / Not Significant"
        
        print(f"{name}:")
        print(f"  Accuracy : {mean_acc:.4f} ± {std_acc:.4f}")
        print(f"  95% CI   : [{ci_lower:.4f}, {ci_upper:.4f}]")
        print(f"  p-value  : {p_val:.5e} ({sig})\n")

In [6]:
run_evaluation('./cell_images')

Extracting features from ./cell_images...


Class: Uninfected: 100%|██████████| 13779/13779 [00:16<00:00, 828.99it/s]



Starting Bayesian Optimization & 20-Fold Evaluation...

[SVM] Optimizing parameters...
[SVM] Best Params Found: OrderedDict({'classifier__C': 2.1602177830877722, 'classifier__gamma': 'auto', 'classifier__kernel': 'rbf'})
[SVM] Running 20-Fold CV on optimized model...

[HistGradientBoosting] Optimizing parameters...
[HistGradientBoosting] Best Params Found: OrderedDict({'classifier__learning_rate': 0.10970919052074331, 'classifier__max_depth': 8, 'classifier__max_iter': 234})
[HistGradientBoosting] Running 20-Fold CV on optimized model...

[Random Forest] Optimizing parameters...
[Random Forest] Best Params Found: OrderedDict({'classifier__max_depth': 17, 'classifier__min_samples_split': 9, 'classifier__n_estimators': 126})
[Random Forest] Running 20-Fold CV on optimized model...

   FINAL OPTIMIZED MODEL STABILITY & 95% CI RESULTS   
SVM:
  Accuracy : 0.9492 ± 0.0064
  95% CI   : [0.9462, 0.9522]
  p-value  : 4.69192e-37 (Stable & Significant)

HistGradientBoosting:
  Accuracy : 0.948

In [7]:
def run_evaluation(root_path):
    import numpy as np
    import joblib
    from scipy import stats
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.model_selection import StratifiedKFold, cross_validate
    from sklearn.svm import SVC
    from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
    from skopt import BayesSearchCV
    from skopt.space import Real, Integer, Categorical

    # 1. Load Data
    X, y = load_data(root_path)

    # 2. Models + Search Spaces
    models_and_spaces = {
        "SVM": (
            SVC(),
            {
                'classifier__C': Real(1e-1, 1e+2, prior='log-uniform'),
                'classifier__gamma': Categorical(['scale', 'auto']),
                'classifier__kernel': Categorical(['rbf', 'linear'])
            }
        ),
        "HistGradientBoosting": (
            HistGradientBoostingClassifier(),
            {
                'classifier__learning_rate': Real(0.01, 0.2, prior='log-uniform'),
                'classifier__max_depth': Integer(3, 15),
                'classifier__max_iter': Integer(50, 400)
            }
        ),
        "Random Forest": (
            RandomForestClassifier(),
            {
                'classifier__n_estimators': Integer(50, 300),
                'classifier__max_depth': Integer(3, 20),
                'classifier__min_samples_split': Integer(2, 10)
            }
        )
    }

    # 3. Cross-validation setup
    inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    outer_cv = StratifiedKFold(n_splits=20, shuffle=True, random_state=42)

    results = {}

    # ✅ Track best model
    best_overall_model = None
    best_score = -np.inf
    best_model_name = None

    print("\nStarting Bayesian Optimization & 20-Fold Evaluation...")

    for name, (model, search_space) in models_and_spaces.items():
        print(f"\n[{name}] Optimizing parameters...")

        pipeline = Pipeline([
            ('scaler', StandardScaler()),
            ('classifier', model)
        ])

        # Bayesian Optimization
        bayes_search = BayesSearchCV(
            estimator=pipeline,
            search_spaces=search_space,
            n_iter=15,
            cv=inner_cv,
            scoring='accuracy',
            random_state=42,
            n_jobs=-1
        )

        bayes_search.fit(X, y)
        best_pipeline = bayes_search.best_estimator_

        print(f"[{name}] Best Params: {bayes_search.best_params_}")
        print(f"[{name}] Running 20-Fold CV...")

        # 20-Fold Evaluation
        cv_results = cross_validate(
            best_pipeline, X, y,
            cv=outer_cv,
            scoring='accuracy',
            n_jobs=-1
        )

        scores = cv_results['test_score']
        results[name] = scores

        mean_acc = np.mean(scores)

        # ✅ Update best model
        if mean_acc > best_score:
            best_score = mean_acc
            best_overall_model = best_pipeline
            best_model_name = name

    # 4. Statistical Analysis
    print("\n=======================================================")
    print("   FINAL OPTIMIZED MODEL STABILITY & 95% CI RESULTS   ")
    print("=======================================================")

    baseline_chance = 0.50

    for name, scores in results.items():
        n_folds = len(scores)
        mean_acc = np.mean(scores)
        std_acc = np.std(scores, ddof=1)

        ci_lower, ci_upper = stats.t.interval(
            confidence=0.95,
            df=n_folds - 1,
            loc=mean_acc,
            scale=stats.sem(scores)
        )

        t_stat, p_val = stats.ttest_1samp(
            scores,
            popmean=baseline_chance,
            alternative='greater'
        )

        sig = "Stable & Significant" if p_val < 0.05 else "Unstable / Not Significant"

        print(f"{name}:")
        print(f"  Accuracy : {mean_acc:.4f} ± {std_acc:.4f}")
        print(f"  95% CI   : [{ci_lower:.4f}, {ci_upper:.4f}]")
        print(f"  p-value  : {p_val:.5e} ({sig})\n")

    # 5. Save Best Model
    print("=======================================================")
    print(f"Best Overall Model: {best_model_name} ({best_score:.4f})")

    # Optional: retrain on full data
    best_overall_model.fit(X, y)

    joblib.dump(best_overall_model, "best_model.pkl", compress=3)

    print("Best model saved as 'best_model.pkl'")

In [8]:
run_evaluation('./cell_images')

Extracting features from ./cell_images...


Class: Uninfected: 100%|██████████| 13779/13779 [00:14<00:00, 979.49it/s] 



Starting Bayesian Optimization & 20-Fold Evaluation...

[SVM] Optimizing parameters...
[SVM] Best Params: OrderedDict({'classifier__C': 2.1602177830877722, 'classifier__gamma': 'auto', 'classifier__kernel': 'rbf'})
[SVM] Running 20-Fold CV...

[HistGradientBoosting] Optimizing parameters...
[HistGradientBoosting] Best Params: OrderedDict({'classifier__learning_rate': 0.12562518673121412, 'classifier__max_depth': 11, 'classifier__max_iter': 395})
[HistGradientBoosting] Running 20-Fold CV...

[Random Forest] Optimizing parameters...
[Random Forest] Best Params: OrderedDict({'classifier__max_depth': 13, 'classifier__min_samples_split': 8, 'classifier__n_estimators': 140})
[Random Forest] Running 20-Fold CV...

   FINAL OPTIMIZED MODEL STABILITY & 95% CI RESULTS   
SVM:
  Accuracy : 0.9492 ± 0.0064
  95% CI   : [0.9462, 0.9522]
  p-value  : 4.69192e-37 (Stable & Significant)

HistGradientBoosting:
  Accuracy : 0.9485 ± 0.0063
  95% CI   : [0.9455, 0.9515]
  p-value  : 3.79848e-37 (Stable 